# 응용 모의고사 Set 5 — 정답 — 그룹별 상관과 군집·회귀

- 데이터: `card_cust.csv`
- 난이도: 기존 Set 01~06과 유사
- 구성: **공통 전처리 → Q1 통계 → Q2 상관분석 → Q3 모델링**
- 모든 문항은 공통 전처리 결과를 이어서 사용합니다.
- 전처리 완료 후 데이터는 **980행**이어야 합니다. 행 수가 다르면 다음 문제로 넘어가기 전에 전처리를 확인하세요.

정답 노트북은 `../answers/`에 있습니다.

## 공통 전처리 정답

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('../../dataset/card_cust.csv')
base = df.copy()
base['MINIMUM_PAYMENTS'] = base['MINIMUM_PAYMENTS'].fillna(
    base['MINIMUM_PAYMENTS'].mean()
)
base = base.loc[base['TENURE'] >= 8].copy()
base['payment_ratio'] = base['PAYMENTS'] / (base['BALANCE'] + 1)
assert len(base) == 980
display(base.head())

## Q1 정답

In [ ]:
corr_by_tenure = base.groupby('TENURE').apply(
    lambda group: group['PAYMENTS'].corr(group['CREDIT_LIMIT'])
)
answer_tenure_q1 = corr_by_tenure.abs().idxmax()
answer_coef_q1 = round(corr_by_tenure.loc[answer_tenure_q1], 2)
display(corr_by_tenure, answer_tenure_q1, answer_coef_q1)  # 10, 0.93

## Q2 정답

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

X = base.drop(columns='CUST_ID')
X_scaled = StandardScaler().fit_transform(X)
scores, labels_by_k = {}, {}
for k in [2, 3, 4, 5]:
    labels = KMeans(n_clusters=k, random_state=321, n_init=10).fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)
    labels_by_k[k] = labels
best_k = pd.Series(scores).idxmax()
clustered = base.copy()
clustered['cluster'] = labels_by_k[best_k]
cash_mean = clustered.groupby('cluster')['CASH_ADVANCE'].mean()
answer_q2 = round(cash_mean.max(), 2)
display(pd.Series(scores), best_k, cash_mean, answer_q2)  # k=2, 1332.76

## Q3 정답

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor

train = base.loc[base['CUST_ID'] % 5 != 0].copy()
test = base.loc[base['CUST_ID'] % 5 == 0].copy()
features = base.columns.difference(['CUST_ID', 'PURCHASES'])
model = DecisionTreeRegressor(random_state=321)
model.fit(train[features], train['PURCHASES'])
pred = model.predict(test[features])
answer_q3 = round(mean_squared_error(test['PURCHASES'], pred) ** 0.5, 2)
display(answer_q3)  # 569.59